LOADING THE DATASET 

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")

print(df.shape)
df.head()

(201664, 25)


,datetime,date,year,month,day,hour,day_of_week,is_weekend,season,city,...,no2,so2,co,o3,temperature,humidity,wind_speed,visibility,aqi,aqi_category
0,2020-01-01 06:00:00,2020-01-01,2020,1,1,6,Wednesday,0,winter,Delhi,...,119.6,47.7,5.19,12.3,9.4,100,3.6,1.2,500,Severe
1,2020-01-01 12:00:00,2020-01-01,2020,1,1,12,Wednesday,0,winter,Delhi,...,117.9,39.3,4.32,15.8,20.6,50,5.9,1.4,500,Severe
2,2020-01-01 18:00:00,2020-01-01,2020,1,1,18,Wednesday,0,winter,Delhi,...,150.1,36.3,7.13,14.3,12.4,56,4.5,1.1,500,Severe
3,2020-01-01 23:00:00,2020-01-01,2020,1,1,23,Wednesday,0,winter,Delhi,...,142.0,30.3,4.90,13.2,14.4,48,5.8,1.4,500,Severe
4,2020-01-01 06:00:00,2020-01-01,2020,1,1,6,Wednesday,0,winter,Delhi,...,138.4,41.5,7.56,15.4,6.8,100,2.8,0.4,500,Severe


In [3]:
df.info()
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201664 entries, 0 to 201663
Data columns (total 25 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   datetime      201664 non-null  object 
 1   date          201664 non-null  object 
 2   year          201664 non-null  int64  
 3   month         201664 non-null  int64  
 4   day           201664 non-null  int64  
 5   hour          201664 non-null  int64  
 6   day_of_week   201664 non-null  object 
 7   is_weekend    201664 non-null  int64  
 8   season        201664 non-null  object 
 9   city          201664 non-null  object 
 10  station       201664 non-null  object 
 11  latitude      201664 non-null  float64
 12  longitude     201664 non-null  float64
 13  pm25          201664 non-null  float64
 14  pm10          201664 non-null  float64
 15  no2           201664 non-null  float64
 16  so2           201664 non-null  float64
 17  co            201664 non-null  float64
 18  o3  

CONVERTING THE DATETIME AND DATE COLUMN FROM OBJECT INTO DATETIME FORMAT 

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("datetime").reset_index(drop=True)

# Filter Delhi only
df = df[df["city"] == "Delhi"].copy()
df = df.sort_values("datetime").reset_index(drop=True)

In [5]:
df = df.drop(columns=["aqi_category", "station", "city"])

In [6]:
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month
df["weekday"] = df["datetime"].dt.weekday

In [7]:
lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]

for lag in lags:
    df[f"AQI_lag_{lag}"] = df["aqi"].shift(lag)

df["AQI_diff_1"] = df["aqi"] - df["AQI_lag_1"]
df["AQI_diff_24"] = df["aqi"] - df["AQI_lag_24"]

df["AQI_pct_change_1"] = df["aqi"].pct_change(1)
df["AQI_pct_change_24"] = df["aqi"].pct_change(24)

In [8]:
pollutants = ["pm25", "pm10", "no2", "so2", "co", "o3"]

for col in pollutants:
    for lag in [1, 3, 6]:
        df[f"{col}_lag_{lag}"] = df[col].shift(lag)

In [9]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dow_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

In [10]:
df["pm25_no2"] = df["pm25"] * df["no2"]
df["pm25_o3"] = df["pm25"] * df["o3"]

In [11]:
df["AQI_roll_3"] = df["aqi"].rolling(3).mean().shift(1)
df["AQI_roll_6"] = df["aqi"].rolling(6).mean().shift(1)
df["AQI_roll_12"] = df["aqi"].rolling(12).mean().shift(1)

df["AQI_std_3"] = df["aqi"].rolling(3).std().shift(1)
df["AQI_std_6"] = df["aqi"].rolling(6).std().shift(1)

In [12]:
df = pd.get_dummies(df, columns=["season"], drop_first=True)

In [13]:
df["target_AQI"] = df["aqi"].shift(-1)

In [14]:
df = df.dropna()

In [15]:
X = df.drop(columns=["datetime", "aqi", "target_AQI", "date","day_of_week"])
y = df["target_AQI"]

In [16]:
print("Object columns:", X.select_dtypes(include=["object"]).columns)
print("Null values:", X.isnull().sum().sum())

Object columns: Index([], dtype='object')
Null values: 0


In [ ]:
#df.to_csv('cleaned_dataset')

In [18]:
df.head()

,datetime,date,year,month,day,hour,day_of_week,is_weekend,latitude,longitude,...,pm25_o3,AQI_roll_3,AQI_roll_6,AQI_roll_12,AQI_std_3,AQI_std_6,season_post_monsoon,season_summer,season_winter,target_AQI
168,2020-01-04 06:00:00,2020-01-04,2020,1,4,6,Saturday,1,28.6999,77.1653,...,4468.80,494.0,497.0,494.916667,10.392305,7.348469,False,False,True,500.0
169,2020-01-04 06:00:00,2020-01-04,2020,1,4,6,Saturday,1,28.7299,77.1718,...,5922.75,494.0,497.0,496.000000,10.392305,7.348469,False,False,True,500.0
170,2020-01-04 06:00:00,2020-01-04,2020,1,4,6,Saturday,1,28.7762,77.0510,...,5613.93,494.0,497.0,496.000000,10.392305,7.348469,False,False,True,500.0
171,2020-01-04 06:00:00,2020-01-04,2020,1,4,6,Saturday,1,28.6469,77.3164,...,5302.88,500.0,497.0,496.000000,0.000000,7.348469,False,False,True,430.0
172,2020-01-04 06:00:00,2020-01-04,2020,1,4,6,Saturday,1,28.6362,77.2010,...,5150.11,500.0,497.0,496.000000,0.000000,7.348469,False,False,True,500.0


In [19]:
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [24]:
"""model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)"""
from sklearn.ensemble import RandomForestRegressor
model=RandomForestRegressor(n_estimators=200)
model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
pred=model.predict(X_test)

In [26]:
from sklearn.metrics import r2_score
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2_score=r2_score(y_test,pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2_score)

MAE: 24.118199412652444
RMSE: 33.499142463527264
R2: 0.9648710162811592
